In [1]:
!pip install transformers jiwer librosa soundfile torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.1 MB/s eta 0:00:00:00:01


In [5]:
import os
import torch
import time
import re
import librosa
from transformers import pipeline
from jiwer import wer

# 1. Khởi tạo phần cứng và mô hình
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng phần cứng: {device}")

# Bạn có thể thay đổi model ở đây để test tốc độ
model_name = "openai/whisper-small" 

pipe = pipeline(
    "automatic-speech-recognition",
    model=model_name,
    device=device,
    chunk_length_s=30
)

# 2. Cấu hình đường dẫn dữ liệu 
base_path = "/kaggle/input/datasets/tuannguyenvananh/vivos-dataset/vivos" 
test_path = os.path.join(base_path, "test")
prompts_file = os.path.join(test_path, "prompts.txt")
waves_dir = os.path.join(test_path, "waves")

# 3. Đọc dữ liệu chuẩn 
ground_truths = {}
with open(prompts_file, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(' ', 1)
        if len(parts) == 2:
            audio_id, transcript = parts
            ground_truths[audio_id] = transcript.lower()

# Hàm xóa dấu câu
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text) 
    return text

# 4. Quá trình kiểm thử 
predictions = []
references = []
count = 0
max_test = 20 

total_audio_duration = 0.0   # Tổng độ dài các file âm thanh
total_inference_time = 0.0   # Tổng thời gian mô hình chạy

print(f"\nBắt đầu nhận diện với model: {model_name}")
for speaker in os.listdir(waves_dir):
    speaker_dir = os.path.join(waves_dir, speaker)
    if not os.path.isdir(speaker_dir):
        continue

    for audio_file in os.listdir(speaker_dir):
        if audio_file.endswith(".wav"):
            audio_id = audio_file.replace(".wav", "")
            
            if audio_id not in ground_truths:
                continue
                
            audio_path = os.path.join(speaker_dir, audio_file)
            
            # Đọc file bằng librosa và đưa về tần số lấy mẫu 16kHz
            speech, sample_rate = librosa.load(audio_path, sr=16000)
            
            # Tính độ dài gốc của file audio
            file_duration = len(speech) / sample_rate
            total_audio_duration += file_duration
            
            # Bắt đầu bấm giờ thời gian mô hình xử lý
            start_time = time.time()
            
            result = pipe(speech, generate_kwargs={"language": "vietnamese"})
            
            # Kết thúc bấm giờ
            end_time = time.time()
            
            inference_time = end_time - start_time
            total_inference_time += inference_time
            
            pred_text = clean_text(result["text"])
            
            predictions.append(pred_text)
            references.append(ground_truths[audio_id])
            
            print(f"[{count+1}] Audio ID: {audio_id}")
            print(f"Độ dài audio: {file_duration:.2f}s | Thời gian AI dịch: {inference_time:.2f}s")
            print(f"Chuẩn: {ground_truths[audio_id]}")
            print(f"AI   : {pred_text}\n")
            
            count += 1
            if max_test and count >= max_test:
                break
    if max_test and count >= max_test:
        break

# 5. Đánh giá % lỗi (WER) và Thống kê thời gian
if len(predictions) > 0:
    error_rate = wer(references, predictions)
    avg_audio_duration = total_audio_duration / count
    avg_inference_time = total_inference_time / count
    
    print("="*50)
    print(f"TỔNG KẾT (Model: {model_name})")
    print(f"==> Tỷ lệ lỗi (WER) trên {count} mẫu: {error_rate * 100:.2f}%")
    print(f"==> Độ dài trung bình mỗi audio: {avg_audio_duration:.2f} giây")
    print(f"==> Tốc độ xử lý trung bình của AI: {avg_inference_time:.2f} giây/file")
    print("="*50)
else:
    print("Lỗi: Không tìm thấy file dữ liệu nào hợp lệ!")

Đang sử dụng phần cứng: cuda:0


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



Bắt đầu nhận diện với model: openai/whisper-small
[1] Audio ID: VIVOSDEV19_196
Độ dài audio: 5.72s | Thời gian AI dịch: 0.77s
Chuẩn: người anh sau đó được phép thầu xây kênh này nhưng không tiến hành
AI   : người ăn sau đó được phép thầu xây kênh này nhưng không tiếng hành

[2] Audio ID: VIVOSDEV19_020
Độ dài audio: 4.69s | Thời gian AI dịch: 0.66s
Chuẩn: lên các cuộc hẹn phỏng vấn và thực hiện phỏng vấn
AI   : lên các cuộc hẹn phong phóng và thực hiện phóng phóng

[3] Audio ID: VIVOSDEV19_030
Độ dài audio: 3.59s | Thời gian AI dịch: 0.74s
Chuẩn: phù sa bồi đắp ruộng vườn cho xóm làng trù phú
AI   : phụ xe bò đắp rụng vườn cho sớm làn trụ phú

[4] Audio ID: VIVOSDEV19_262
Độ dài audio: 4.66s | Thời gian AI dịch: 0.66s
Chuẩn: để rồi chúng ta khao khát mãi trong vòng luân hồi nhân gian
AI   : để rồi chúng ta khau khác mãi trong vòng luôn hồi nhanh gian

[5] Audio ID: VIVOSDEV19_064
Độ dài audio: 6.62s | Thời gian AI dịch: 0.82s
Chuẩn: chóng mặt còn có thể do tình trạng rối loạn nhịp tim